# Merge Tigrinya ASR Datasets → One Hugging Face Dataset

This notebook combines three Tigrinya speech datasets into a single unified ASR corpus and pushes it to your Hugging Face Hub account.

Source datasets:
- `badrex/tigrinya-speech`
- `google/WaxalNLP` (config `tir_asr`)
- `UBC-NLP/SimbaBench_dataset` (config `asr_test_tir`)

**Important:** `SimbaBench_dataset`'s `asr_test_tir` config looks like a held-out benchmark test set, not training data — its name literally says "test". To avoid leaking benchmark data into training, this notebook keeps it **separate**: `badrex` + `WaxalNLP` get merged and re-split into train/validation, while SimbaBench is kept intact as its own untouched test split. Set `MERGE_SIMBABENCH_INTO_TRAINING = True` in section 3 if you'd rather treat it as ordinary training data instead.

**Workflow:**
1. Install deps & log in to Hugging Face
2. Load each dataset and **inspect its raw schema** (column names — and configs — differ across datasets, so don't skip this)
3. Standardize each one to a common schema: `audio`, `text`, `source`
4. Concatenate the training-eligible sources
5. Normalize Tigrinya text, resample audio, drop empty/broken rows
6. Deduplicate (by transcript text)
7. Re-split into train/validation, append SimbaBench as `test`
8. Sanity check + audio-hours check
9. Push the merged dataset to the Hub, with a dataset card

> ⚠️ Before pushing publicly, check each source dataset's license/card to confirm redistribution as a merged dataset is allowed. Most CC-BY style licenses are fine, but always verify — and double check SimbaBench's license/terms specifically, since benchmark datasets sometimes carry stricter redistribution terms than plain speech corpora.

## 1. Setup

In [1]:
!pip install -q -U datasets huggingface_hub soundfile librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [2]:
from huggingface_hub import notebook_login

# Paste a HF token with WRITE access when prompted (huggingface.co/settings/tokens)
notebook_login()

In [3]:
from datasets import load_dataset, concatenate_datasets, Audio, DatasetDict, Dataset
import re
import unicodedata

# Each entry is (repo_id, config_name). config_name=None means single-config / no config needed.
SOURCE_REPOS = [
    ("badrex/tigrinya-speech", None),
    ("google/WaxalNLP", "tir_asr"),
    ("UBC-NLP/SimbaBench_dataset", "asr_test_tir"),
]

## 2. Inspect each dataset's schema

Run this first and actually read the output — column names (e.g. `text` vs `transcript` vs `sentence`) and split names vary across these repos. You'll use what you see here to fill in the `COLUMN_MAP` in the next section.

In [4]:
from datasets import get_dataset_config_names, get_dataset_split_names

for repo, known_config in SOURCE_REPOS:
    print(f"\n{'='*60}\n{repo}  (expected config: {known_config})\n{'='*60}")
    try:
        configs = get_dataset_config_names(repo)
        print("available configs:", configs)
        for cfg in configs:
            splits = get_dataset_split_names(repo, cfg)
            print(f"  config={cfg} splits={splits}")
    except Exception as e:
        print("Could not list configs/splits:", e)
        continue

    try:
        cfg_to_use = known_config if known_config else configs[0]
        splits = get_dataset_split_names(repo, cfg_to_use)
        first_split = splits[0]
        # streaming=True avoids downloading the whole config's data folder (all splits,
        # including any huge "unlabeled" split) just to preview one row.
        preview_stream = load_dataset(repo, cfg_to_use, split=first_split, streaming=True)
        row = next(iter(preview_stream))
        print(f"features (config={cfg_to_use}):", preview_stream.features)
        print("example keys:", list(row.keys()))
    except Exception as e:
        print("Could not preview:", e)



badrex/tigrinya-speech  (expected config: None)


README.md:   0%|          | 0.00/826 [00:00<?, ?B/s]

available configs: ['default']
  config=default splits=['validation', 'test', 'train']
features (config=default): {'audio_id': Value('string'), 'speaker_id': Value('string'), 'audio': Audio(sampling_rate=16000, decode=True, num_channels=None, stream_index=None), 'audio_duration': Value('float64'), 'transcription': Value('string'), 'gender': Value('string'), 'age_group': Value('string'), 'locale': Value('string')}
example keys: ['audio_id', 'speaker_id', 'audio', 'audio_duration', 'transcription', 'gender', 'age_group', 'locale']

google/WaxalNLP  (expected config: tir_asr)


README.md:   0%|          | 0.00/31.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/79 [00:00<?, ?it/s]

  config=asr_test_zul splits=['test']


Resolving data files:   0%|          | 0/79 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/79 [00:00<?, ?it/s]

features (config=asr_test_tir): {'split': Value('string'), 'benchmark_id': Value('string'), 'audio': Audio(sampling_rate=16000, decode=True, num_channels=None, stream_index=None), 'text': Value('string'), 'duration_s': Value('float32'), 'lang_iso3': Value('string'), 'lang_name': Value('string')}
example keys: ['split', 'benchmark_id', 'audio', 'text', 'duration_s', 'lang_iso3', 'lang_name']


## 3. Configure column mapping

Update `COLUMN_MAP` below based on what the inspection cell printed for each repo. Each entry maps `repo_id -> {"config": <config name or None>, "text": <transcript column name>, "audio": <audio column name>, "splits": [<split names to include>], "role": "train_pool" or "held_out_test"}`.

The defaults below are my best guess as of writing — **verify against your own inspection output above before running the merge**, since column names and configs can change.

`"role"` controls what happens to each source later:
- `"train_pool"`: gets merged with the other train_pool sources and re-split into train/validation in section 8.
- `"held_out_test"`: kept completely separate, never shuffled together with training data, and becomes (or is appended to) the final `test` split as-is.

SimbaBench defaults to `"held_out_test"` because its config name (`asr_test_tir`) signals it's a benchmark set. Change it to `"train_pool"` below if you want it treated as ordinary training data instead.

In [5]:
import requests

def check_schema(repo_id):
    url = f"https://datasets-server.huggingface.co/info?dataset={repo_id}"
    r = requests.get(url).json()
    if "error" in r:
        print(f"{repo_id}: ERROR -> {r['error']}")
        return
    for config, info in r["dataset_info"].items():
        print(f"\n{repo_id} [{config}]")
        for feat_name, feat_type in info["features"].items():
            print(f"  {feat_name}: {feat_type.get('_type') or feat_type.get('dtype')}")
        print(f"  splits: {list(info['splits'].keys())}")

for repo in [
    "badrex/tigrinya-speech",
    "google/WaxalNLP",
    "UBC-NLP/SimbaBench_dataset",
]:
    check_schema(repo)


badrex/tigrinya-speech [default]
  audio_id: Value
  speaker_id: Value
  audio: Audio
  audio_duration: Value
  transcription: Value
  gender: Value
  age_group: Value
  locale: Value
  splits: ['validation', 'test', 'train']

UBC-NLP/SimbaBench_dataset [asr_test_Akuapim-twi]
  split: Value
  benchmark_id: Value
  audio: Audio
  text: Value
  duration_s: Value
  lang_iso3: Value
  lang_name: Value
  splits: ['test']

UBC-NLP/SimbaBench_dataset [asr_test_Asante-twi]
  split: Value
  benchmark_id: Value
  audio: Audio
  text: Value
  duration_s: Value
  lang_iso3: Value
  lang_name: Value
  splits: ['test']

UBC-NLP/SimbaBench_dataset [asr_test_afr]
  split: Value
  benchmark_id: Value
  audio: Audio
  text: Value
  duration_s: Value
  lang_iso3: Value
  lang_name: Value
  splits: ['test']

UBC-NLP/SimbaBench_dataset [asr_test_amh]
  split: Value
  benchmark_id: Value
  audio: Audio
  text: Value
  duration_s: Value
  lang_iso3: Value
  lang_name: Value
  splits: ['test']

UBC-NLP/Simba

In [6]:
COLUMN_MAP = {
    "badrex/tigrinya-speech": {
        "config": None,
        "text": "transcription",
        "audio": "audio",
        "splits": None,            # confirmed splits: validation, test, train — no unlabeled here, so None is fine
        "role": "train_pool",
        "force_streaming": False,  # clean parquet dataset, no bundled-split issue — non-streaming is fast and safe
    },
    "google/WaxalNLP": {
        "config": "tir_asr",
        "text": "transcription",
        "audio": "audio",
        "splits": ["train", "validation", "test"],  # explicitly exclude "unlabeled"
        "role": "train_pool",
        "force_streaming": True,   # builder bundles ALL splits (incl. huge "unlabeled") into one download
                                     # when loaded non-streaming, even if you only ask for "train" — streaming
                                     # avoids pulling files for splits you didn't request.
    },
    "UBC-NLP/SimbaBench_dataset": {
        "config": "asr_test_tir",
        "text": "text",
        "audio": "audio",
        "splits": None,
        "role": "held_out_test",
        "force_streaming": False,  # only 7 rows in this split — irrelevant either way, but non-streaming is fine
    },
}


## 4. Load + standardize each dataset

Every dataset gets reduced to three columns: `audio`, `text`, `source`. Results are kept in two separate buckets according to each source's `"role"`.

Loading uses the fast non-streaming path by default. Only sources flagged `"force_streaming": True` in `COLUMN_MAP` use streaming instead — currently just `google/WaxalNLP`, whose builder bundles every split (including the large `unlabeled` one) into one download even if you only ask for `train`.

For streamed sources, rows are converted with `Dataset.from_generator()` rather than `Dataset.from_list(list(stream))`. The generator form writes to disk incrementally as it iterates, so memory usage stays roughly flat regardless of split size — important here since a ~50k-row audio split held entirely in a Python list (with every clip decoded) can be tens of GB and crash the runtime.


In [7]:
def load_and_standardize(repo_id, cfg):
    config_name = cfg["config"]
    text_col = cfg["text"]
    audio_col = cfg["audio"]
    splits = cfg["splits"]
    use_streaming = cfg.get("force_streaming", False)

    if splits is None:
        splits = get_dataset_split_names(repo_id, config_name) if config_name else get_dataset_split_names(repo_id)

    per_split = []
    for split in splits:
        if use_streaming:
            # Only used for sources whose builder bundles unrelated splits into one download
            # (see the "force_streaming" note in COLUMN_MAP).
            #
            # IMPORTANT: we use Dataset.from_generator() here, NOT Dataset.from_list(list(stream)).
            # from_list would materialize every row (including decoded audio arrays) in a Python
            # list in RAM before writing anything to disk — for a ~50k row audio split that can be
            # tens of GB and crash the runtime. from_generator writes to disk incrementally as it
            # iterates, so memory stays roughly constant regardless of split size.
            def row_generator():
                stream = load_dataset(repo_id, config_name, split=split, streaming=True) if config_name \
                    else load_dataset(repo_id, split=split, streaming=True)
                for row in stream:
                    yield row

            ds = Dataset.from_generator(row_generator)
        else:
            # Fast path: bulk parquet download + native arrow conversion.
            ds = load_dataset(repo_id, config_name, split=split) if config_name \
                else load_dataset(repo_id, split=split)

        rename_map = {}
        if text_col != "text":
            rename_map[text_col] = "text"
        if audio_col != "audio":
            rename_map[audio_col] = "audio"
        if rename_map:
            ds = ds.rename_columns(rename_map)

        # Keep only the columns we need
        cols_to_drop = [c for c in ds.column_names if c not in ("text", "audio")]
        if cols_to_drop:
            ds = ds.remove_columns(cols_to_drop)

        ds = ds.cast_column("audio", Audio(sampling_rate=16000))
        tag = f"{repo_id}" + (f"/{config_name}" if config_name else "") + f":{split}"
        ds = ds.add_column("source", [tag] * len(ds))
        per_split.append(ds)

    return concatenate_datasets(per_split)


train_pool_datasets = []   # gets merged + re-split into train/validation
held_out_test_datasets = []  # kept intact, becomes/extends the final test split

for repo_id, cfg in COLUMN_MAP.items():
    print(f"Loading {repo_id} (config={cfg['config']}, role={cfg['role']}, streaming={cfg.get('force_streaming', False)}) ...")
    ds = load_and_standardize(repo_id, cfg)
    print(f"  -> {len(ds)} rows")
    if cfg["role"] == "held_out_test":
        held_out_test_datasets.append(ds)
    else:
        train_pool_datasets.append(ds)


Loading badrex/tigrinya-speech (config=None, role=train_pool, streaming=False) ...


data/validation-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  362MB            

data/validation-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/validation-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  342MB            

data/validation-00001-of-00002.parquet: downloading bytes:           |  0.00B            

...

Resolving data files:   0%|          | 0/79 [00:00<?, ?it/s]

asr_test/HF_test-tir*cv19.parquet: reconstructing file:   0%|          |  0.00B / 1.02MB            

asr_test/HF_test-tir*cv19.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/7 [00:00<?, ? examples/s]

  -> 7 rows


## 5. Concatenate the training-eligible sources

Only `train_pool` sources get combined here. `held_out_test` sources (SimbaBench, by default) are left untouched in `held_out_test_datasets` and reintroduced in section 8.

In [8]:
merged = concatenate_datasets(train_pool_datasets)
print(f"Total train-pool rows (before cleaning): {len(merged)}")
if held_out_test_datasets:
    held_out_total = sum(len(d) for d in held_out_test_datasets)
    print(f"Held-out test rows (kept separate): {held_out_total}")
merged

Total train-pool rows (before cleaning): 65095
Held-out test rows (kept separate): 7


Dataset({
    features: ['audio', 'text', 'source'],
    num_rows: 65095
})

## 6. Clean & normalize text

- Strip surrounding whitespace and collapse internal whitespace
- Drop empty transcripts
- Unicode-normalize (NFC) so visually-identical Ge'ez characters are byte-identical across sources

In [9]:
def normalize_text(example):
    text = example["text"] or ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text).strip()
    example["text"] = text
    return example

merged = merged.map(normalize_text)
before = len(merged)
merged = merged.filter(lambda ex: len(ex["text"]) > 0)
print(f"Dropped {before - len(merged)} rows with empty transcripts")
print(f"Remaining rows: {len(merged)}")

Map:   0%|          | 0/65095 [00:00<?, ? examples/s]

Filter:   0%|          | 0/65095 [00:00<?, ? examples/s]

Dropped 0 rows with empty transcripts
Remaining rows: 65095


## 7. Deduplicate

Removes exact-duplicate transcripts within the train pool (common when the same source recordings appear in more than one dataset). Keeps the first occurrence.

It also checks the train pool against the held-out test set (SimbaBench) and drops any train-pool rows whose transcript also appears in the test set — this prevents test-set leakage into training, which matters even more here since SimbaBench is a purpose-built benchmark.

In [10]:
# Dedup within the train pool
seen = set()
keep_indices = []
for i, text in enumerate(merged["text"]):
    key = text.strip().lower()
    if key not in seen:
        seen.add(key)
        keep_indices.append(i)

print(f"Dropping {len(merged) - len(keep_indices)} duplicate-transcript rows (within train pool)")
merged = merged.select(keep_indices)
print(f"Remaining train-pool rows: {len(merged)}")

# Cross-check against held-out test set to avoid leakage
if held_out_test_datasets:
    held_out_texts = set()
    for d in held_out_test_datasets:
        held_out_texts.update(t.strip().lower() for t in d["text"])

    keep_indices = [i for i, text in enumerate(merged["text"]) if text.strip().lower() not in held_out_texts]
    leaked = len(merged) - len(keep_indices)
    if leaked:
        print(f"Dropping {leaked} train-pool rows whose transcript also appears in the held-out test set")
        merged = merged.select(keep_indices)
    print(f"Final train-pool rows: {len(merged)}")

Dropping 14897 duplicate-transcript rows (within train pool)
Remaining train-pool rows: 50198
Final train-pool rows: 50198


## 8. Re-split into train / validation, append held-out test

The train pool (badrex + WaxalNLP, minus anything overlapping with the test set) is shuffled and split 90/10 into train/validation, ignoring each source's original splits. The held-out test set (SimbaBench, by default) is appended as-is — untouched and unshuffled with the rest — so it stays a clean benchmark split.

In [11]:
merged = merged.shuffle(seed=42)

n = len(merged)
train_end = int(n * 0.90)

splits = {
    "train": merged.select(range(0, train_end)),
    "validation": merged.select(range(train_end, n)),
}

if held_out_test_datasets:
    splits["test"] = concatenate_datasets(held_out_test_datasets)

dataset_dict = DatasetDict(splits)
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['audio', 'text', 'source'],
        num_rows: 45178
    })
    validation: Dataset({
        features: ['audio', 'text', 'source'],
        num_rows: 5020
    })
    test: Dataset({
        features: ['audio', 'text', 'source'],
        num_rows: 7
    })
})

## 9. Sanity check

Listen to / inspect a couple of rows before pushing, to make sure the audio and text actually line up correctly.

In [12]:
import IPython.display as ipd

for i in range(3):
    ex = dataset_dict["train"][i]
    print(f"[{ex['source']}] {ex['text']}")
    display(ipd.Audio(ex["audio"]["array"], rate=ex["audio"]["sampling_rate"]))

[badrex/tigrinya-speech:train] ኣብ ትግራይ ዝተፈላለዩ ምሁራትን ደረስትን ሰባት ከምዘለዉ ይፍለጥ እዩ፡፡ እቲ ትሪእዎ ዘለኹም ሓደ ናይ ቅዱስ ያሬድ ዘርኢ ስእሊ እዩ፡፡ እዚ ኣበይ ከባቢ ይርከብ?


[badrex/tigrinya-speech:train] መሰረታዊ ልምዓት ክንብል ከለና ኣብ ሓደ ከባቢ ወይ ኣብ ሓደ ከተማ ክግበሩ ዝክእሉ ጥቅምታት ነቲ ከተማ ንክምዕርግን ፅቡቅ ኣገልግልት ንክህልዎን ዝሕግዙ ነገራት እዮም፡፡ኣብነት ፅርግያ፣ መብራህቲ፣ ማይ እዚኦም ዝምሳሰሉ ምጥቃስ ይከኣል፡፡


[google/WaxalNLP/tir_asr:train] እዚ ኣብ ሓድሽ ናይ ኣፓርታማ ምልክታ፡ ብቐይሕን ቀጠልያን ዝተሸፈነ ዝርዝራት ይበርህ። 50% ቅናስን ‘ሕጂ ደውሉ!’ ዝብል ብኣራንሺ ዝተጻሕፈ ኮይኑ፡ ቁጽሪ ተሌፎን ብየማን ስእሊ ናይ ሓደ ዓቢ ህንጻ ብጽባቐ ይረአ ኣሎ።


## 9b. Total audio duration

Computes total hours per split by decoding audio in memory, batch by batch — it does **not** use `.map()`, since `.map()` writes a whole new Arrow cache file to disk for every row (including the decoded audio), which can fill Colab's disk on a multi-GB dataset. This version discards each batch after summing it, so disk usage stays flat.

In [13]:
# Quick sanity check on remaining disk space before proceeding
!df -h / | tail -1

overlay         108G   66G   43G  61% /


In [14]:
def compute_total_hours(ds, batch_size=500):
    total_seconds = 0.0
    n = len(ds)
    for i in range(0, n, batch_size):
        batch = ds[i:i + batch_size]  # decodes audio for this batch only, in memory
        for audio in batch["audio"]:
            total_seconds += len(audio["array"]) / audio["sampling_rate"]
    return total_seconds / 3600

split_hours = {}
for split_name, split_ds in dataset_dict.items():
    hours = compute_total_hours(split_ds)
    split_hours[split_name] = hours
    print(f"{split_name:>12}: {hours:7.2f} hours   ({len(split_ds):>7} clips,  avg {hours*3600/len(split_ds):.2f}s/clip)")

total_hours_all = sum(split_hours.values())
print(f"{'TOTAL':>12}: {total_hours_all:7.2f} hours   ({sum(len(d) for d in dataset_dict.values()):>7} clips)")

       train:  196.35 hours   (  45178 clips,  avg 15.65s/clip)
  validation:   21.85 hours   (   5020 clips,  avg 15.67s/clip)
        test:    0.01 hours   (      7 clips,  avg 5.20s/clip)
       TOTAL:  218.22 hours   (  50205 clips)


## 10. Push to the Hugging Face Hub

Set `HUB_REPO_ID` to `your-username/dataset-name`. Set `PRIVATE = True` first if you want to review it privately before making it public.

In [15]:
HUB_REPO_ID = "Harbidel/tigrinya-asr-merged"  # <-- change this
PRIVATE = True  # flip to False when you're ready to make it public

dataset_dict.push_to_hub(HUB_REPO_ID, private=PRIVATE)
print(f"Pushed to https://huggingface.co/datasets/{HUB_REPO_ID}")

Uploading the dataset shards:   0%|          | 0/29 [00:00<?, ? shards/s]

Map:   0%|          | 0/1558 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpti2y2jtm.parquet    : 100%|#########9| 1.17MB / 1.17MB            

Pushed to https://huggingface.co/datasets/Harbidel/tigrinya-asr-merged


## 11. (Optional) Write a dataset card

This creates a basic README.md on the Hub repo crediting the four source datasets.

In [16]:
from huggingface_hub import HfApi

# Requires split_hours and total_hours_all from the "Total audio duration" cell above — run that first

readme = f"""---
language:
- am
license: cc-by-4.0
task_categories:
- automatic-speech-recognition
pretty_name: Merged Tigrinya ASR Dataset
---

# {HUB_REPO_ID.split('/')[-1]}

A merged Tigrinya speech-recognition dataset, combining and deduplicating:

- [badrex/tigrinya-speech](https://huggingface.co/datasets/badrex/tigrinya-speech) (train pool)
- [google/WaxalNLP](https://huggingface.co/datasets/google/WaxalNLP) config `tir_asr` (train pool)
- [UBC-NLP/SimbaBench_dataset](https://huggingface.co/datasets/UBC-NLP/SimbaBench_dataset) config `asr_test_tir` (held-out benchmark test set)

## Processing

- Standardized to `audio` (16kHz mono) and `text` columns, with a `source` column tracking origin
- Unicode NFC-normalized transcripts, empty transcripts dropped
- Exact-duplicate transcripts removed from the train pool
- Train-pool rows whose transcript overlaps with the SimbaBench test set were dropped, to prevent test leakage
- Train pool re-split into train (90%) / validation (10%), ignoring original source splits
- SimbaBench's `asr_test_tir` split is kept unmodified as the `test` split

## Rows & duration

{chr(10).join(f"- {name.capitalize()}: {len(dataset_dict[name])} clips ({split_hours[name]:.2f} hours)" for name in dataset_dict.keys())}
- **Total: {sum(len(d) for d in dataset_dict.values())} clips ({total_hours_all:.2f} hours)**

## License

Check the license of each source dataset before redistributing; this card assumes CC-BY-4.0 as the most permissive common denominator among the sources at time of writing — verify this still holds for all four before relying on it.
"""

api = HfApi()
api.upload_file(
    path_or_fileobj=readme.encode(),
    path_in_repo="README.md",
    repo_id=HUB_REPO_ID,
    repo_type="dataset",
)
print("README.md uploaded.")

README.md uploaded.
